In [3]:
!pip install datasets av

# AnyUp 3D Training Notebook

This notebook trains the 3D `AnyUp` model (video feature upsampling).  
It mirrors the structure of `train.py` but is interactive — tweak and re-run cells freely.

**Branch:** `3Dconv`  
**Model:** `anyup.model.AnyUp` (spatiotemporal cross-attention)  
**Teacher:** VideoMAE (or any video encoder producing 5D feature tensors `(B, C, T, H, W)`)

## Setup: Clone Repo (Colab)

Run this cell first — it clones the project from GitHub so all `anyup.*` imports work.  
Safe to re-run: skips the clone if the repo is already present.

In [1]:
import os

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "3Dconv"
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"Repo already present at {REPO_DIR} — skipping clone.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

Repo already present at /content/anyup — skipping clone.
Working directory: /content/anyup


## 0. Imports & Paths

In [2]:
import os
import sys
import random
import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import autocast
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm

# Make sure the repo root is on the path
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Repo root: /content/anyup
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 1. Configuration

All knobs are here. Adjust before running the rest.

> **Colab users:** After `cfg = TrainConfig()` there is a clearly marked override block.
> That block is the only place you need to edit for each new run or new account.

In [3]:
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

@dataclass
class TStage:
    t: int           # number of frames for this stage
    start_step: int  # global step at which this stage activates
    batch_size: int  # batch size to use in this stage

@dataclass
class TrainConfig:
    # --- Experiment identity ---
    # Change this every time you start a new run or switch accounts.
    experiment_name: str = "run_01"

    # --- Paths ---
    model_ckpt_2d: Optional[str] = "checkpoints/anyup_2d.pth"
    checkpoint_dir: str = "checkpoints/anyup3d"
    resume: Optional[str] = None
    log_dir: str = "runs/anyup3d"
    dataset_root: str = "data/videos"

    # --- Video encoder (teacher) ---
    encoder: str = "videomae"
    encoder_model: str = "MCG-NJU/videomae-base"

    # --- Model hyper-params ---
    input_dim: int = 3
    qk_dim: int = 128
    num_heads: int = 4
    window_ratio: float = 0.1
    kernel_size_lfu: int = 5
    t_k: int = 1

    # --- Spatial resolution ---
    img_size: int = 224
    crop_size: int = 448

    # --- T-curriculum ---
    # NOTE: first stage must have t >= tubelet_size (2 for VideoMAE base).
    # t=1 is invalid — VideoMAE uses tubelet embeddings of size 2.
    t_schedule: List[TStage] = field(default_factory=lambda: [
        TStage(t=2,  start_step=0,     batch_size=8),
        TStage(t=4,  start_step=5000,  batch_size=4),
        TStage(t=8,  start_step=15000, batch_size=2),
        TStage(t=16, start_step=30000, batch_size=1),
    ])

    # --- Optimizer ---
    lr: float = 2e-4
    weight_decay: float = 1e-4
    grad_clip_max_norm: float = 1.0

    # --- Loss weights ---
    lambda_cos_mse: float = 1.0
    lambda_self_consistency: float = 0.1
    lambda_temporal_consistency: float = 0.2
    temporal_lambda_warmup_steps: int = 5000

    # --- Training duration ---
    max_steps: int = 50_000
    save_every_n_steps: int = 500
    val_every_n_steps: int = 1000
    log_every_n_steps: int = 50

    # --- Data ---
    num_workers: int = 4
    pin_memory: bool = True

    # --- Misc ---
    seed: int = 42
    debug: bool = False
    debug_steps: int = 10


cfg = TrainConfig()

# ============================================================
# COLAB OVERRIDES — edit this block for every new run.
#
# 1. Change experiment_name each time you start fresh or
#    switch accounts (e.g. "run_01", "run_02").
# 2. DRIVE_ROOT must match your MyDrive folder.
# 3. dataset_root stays '/content/local_dataset' — Cell 7a
#    copies the data there from Drive before training.
# ============================================================

DRIVE_ROOT = "/content/drive/MyDrive/anyup3d"  # <-- change to your Drive folder

cfg.experiment_name = "run_01"                 # <-- change for each new run / account
cfg.model_ckpt_2d   = f"{DRIVE_ROOT}/anyup_2d.pth"
cfg.checkpoint_dir  = f"{DRIVE_ROOT}/checkpoints/{cfg.experiment_name}"
cfg.log_dir         = f"{DRIVE_ROOT}/runs/{cfg.experiment_name}"
cfg.dataset_root    = "/content/local_dataset"  # local copy — see Cell 7a

# cfg.resume = f"{DRIVE_ROOT}/checkpoints/run_01/anyup3d_step5000.pth"  # uncomment to resume
# cfg.debug  = True   # uncomment for a 10-step smoke test

print(cfg)


TrainConfig(experiment_name='run_01', model_ckpt_2d='/content/drive/MyDrive/anyup3d/anyup_2d.pth', checkpoint_dir='/content/drive/MyDrive/anyup3d/checkpoints/run_01', resume=None, log_dir='/content/drive/MyDrive/anyup3d/runs/run_01', dataset_root='/content/local_dataset', encoder='videomae', encoder_model='MCG-NJU/videomae-base', input_dim=3, qk_dim=128, num_heads=4, window_ratio=0.1, kernel_size_lfu=5, t_k=1, img_size=224, crop_size=448, t_schedule=[TStage(t=2, start_step=0, batch_size=8), TStage(t=4, start_step=5000, batch_size=4), TStage(t=8, start_step=15000, batch_size=2), TStage(t=16, start_step=30000, batch_size=1)], lr=0.0002, weight_decay=0.0001, grad_clip_max_norm=1.0, lambda_cos_mse=1.0, lambda_self_consistency=0.1, lambda_temporal_consistency=0.2, temporal_lambda_warmup_steps=5000, max_steps=50000, save_every_n_steps=500, val_every_n_steps=1000, log_every_n_steps=50, num_workers=4, pin_memory=True, seed=42, debug=False, debug_steps=10)


## 2. Reproducibility & Device

In [4]:
import numpy as np

torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

Training on: cuda


## 3. Mount Drive & Logger (TensorBoard)

Drive is mounted here so that checkpoints and TensorBoard logs are written to it immediately.
The free Colab GPU session lasts ~4–5 hours — anything written only to `/content/` is lost
when the session ends.

In [5]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted at /content/drive")
except ModuleNotFoundError:
    print("Not running in Colab — Drive mount skipped.")

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_log_dir = Path(cfg.log_dir) / timestamp
run_log_dir.mkdir(parents=True, exist_ok=True)

writer = SummaryWriter(log_dir=str(run_log_dir))
print(f"TensorBoard log dir: {run_log_dir}")
print("Launch TensorBoard with:")
print(f"  %load_ext tensorboard")
print(f"  %tensorboard --logdir {cfg.log_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted at /content/drive
TensorBoard log dir: /content/drive/MyDrive/anyup3d/runs/run_01/2026-04-16_07-04-40
Launch TensorBoard with:
  %load_ext tensorboard
  %tensorboard --logdir /content/drive/MyDrive/anyup3d/runs/run_01


## 4. Video Encoder (Teacher / GT features)

The teacher extracts ground-truth spatiotemporal features that AnyUp is trained to reproduce.  
Currently wired for **VideoMAE**. Swap this cell to use a different encoder.

In [6]:
# ---- VideoMAE encoder ----
# Output shape: (B, T*H*W, C) spatial tokens -> reshaped to (B, C, T, H, W)
# For ViT-B/16 at 224px: H=W=14, so token grid is (T, 14, 14)

from transformers import VideoMAEModel, AutoImageProcessor

print(f"Loading teacher: {cfg.encoder_model} ...")
processor = AutoImageProcessor.from_pretrained(cfg.encoder_model)
encoder = VideoMAEModel.from_pretrained(cfg.encoder_model).to(device).eval()

# Freeze — we never train the teacher
for p in encoder.parameters():
    p.requires_grad_(False)

# Derive encoder output dim & patch size
ENCODER_DIM = encoder.config.hidden_size     # e.g. 768 for ViT-B
PATCH_SIZE  = encoder.config.patch_size      # e.g. 16
TOKEN_H = cfg.img_size // PATCH_SIZE
TOKEN_W = cfg.img_size // PATCH_SIZE

print(f"Encoder dim={ENCODER_DIM}, patch_size={PATCH_SIZE}, spatial token grid={TOKEN_H}x{TOKEN_W}")

Loading teacher: MCG-NJU/videomae-base ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The image processor of type `VideoMAEImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.


Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.bias                | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED |  | 
decoder.norm.weight                                               

Encoder dim=768, patch_size=16, spatial token grid=14x14


## 5. AnyUp 3D Model

In [7]:
from anyup.model import AnyUp
from scripts.load_2d_weights import build_3d_state_dict

anyup = AnyUp(
    input_dim=cfg.input_dim,
    qk_dim=cfg.qk_dim,
    num_heads=cfg.num_heads,
    window_ratio=cfg.window_ratio,
    kernel_size_lfu=cfg.kernel_size_lfu,
    t_k=cfg.t_k,
).to(device)

# --- Optional: load 2D pretrained weights ---
if cfg.model_ckpt_2d and Path(cfg.model_ckpt_2d).exists():
    print(f"Loading 2D checkpoint: {cfg.model_ckpt_2d}")
    raw = torch.load(cfg.model_ckpt_2d, map_location="cpu", weights_only=False)
    sd = raw
    for key in ("anyup", "model", "state_dict", "weights", "params"):
        if isinstance(sd, dict) and key in sd:
            sd = sd[key]
            break
    adapted_sd, status = build_3d_state_dict(sd, anyup)
    anyup.load_state_dict(adapted_sd, strict=True)
    print(f"  Loaded with shape adaptation ({len(adapted_sd)} tensors)")
elif cfg.model_ckpt_2d:
    print(f"[WARNING] 2D checkpoint not found at '{cfg.model_ckpt_2d}' — starting from random init")
else:
    print("Starting from random init (no 2D checkpoint configured)")

# --- Optional: resume 3D checkpoint ---
global_step = 0
data_epoch  = 0   # tracks how many full passes through the dataset we've done;
                  # used to seed the DataLoader so each pass has a different shuffle order
if cfg.resume and Path(cfg.resume).exists():
    print(f"Resuming from: {cfg.resume}")
    ckpt = torch.load(cfg.resume, map_location="cpu", weights_only=False)
    anyup.load_state_dict(ckpt["anyup"])
    global_step = ckpt.get("global_step", 0)
    data_epoch  = ckpt.get("data_epoch",  0)  # restore shuffle position
    print(f"  Resumed at step {global_step}, data_epoch {data_epoch}")

anyup.train()
num_params = sum(p.numel() for p in anyup.parameters() if p.requires_grad)
print(f"AnyUp 3D trainable params: {num_params:,}")

Loading 2D checkpoint: /content/drive/MyDrive/anyup3d/anyup_2d.pth
  Loaded with shape adaptation (73 tensors)
AnyUp 3D trainable params: 878,208


## 6. Loss Functions

In [8]:
from anyup.loss import Cosine_MSE

criterion = Cosine_MSE()

def cosine_mse_3d(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """
    pred, target: (B, C, T, H, W)  ->  flatten T,H,W then call 2D criterion.
    Returns scalar loss.
    """
    B, C, T, H, W = pred.shape
    pred_2d   = pred.view(B, C, T * H, W)
    target_2d = target.view(B, C, T * H, W)
    return criterion(pred_2d, target_2d)["total"]


def get_temporal_lambda(step: int) -> float:
    """Linearly ramp temporal consistency weight from 0 to cfg value."""
    warmup = cfg.temporal_lambda_warmup_steps
    if warmup <= 0:
        return cfg.lambda_temporal_consistency
    return cfg.lambda_temporal_consistency * min(1.0, step / warmup)


print("Loss functions ready.")

Loss functions ready.


## 7. Dataset — Kinetics-400

Downloads ~55 GB of Kinetics-400 clips directly to `/content/kinetics_subset` (Colab local disk).
No Drive space needed.

**Each new session:** re-run this cell — it resumes from however many clips are already there.

> First time only: `!pip install datasets av`


In [13]:
from pathlib import Path
import torch
import torchvision.transforms.v2 as Tv2
from torchvision.transforms import InterpolationMode
from torch.utils.data import DataLoader, Dataset
import av
import random as _random

# 1. Grab datasets from HF
try:
    from datasets import load_dataset
except ImportError:
    !pip install -q datasets
    from datasets import load_dataset

# 2. Setup folders
TEST_DIR = "/content/test_videos"
Path(TEST_DIR).mkdir(parents=True, exist_ok=True)

print("🚀 Downloading a small video subset to test your pipeline...")
ds = load_dataset("sayakpaul/ucf101-subset", split="train")

# Let's download the first 10 video files into our directory
import urllib.request
for i in range(10):
    sample = ds[i]
    # The dataset contains URLs to video files
    url = sample['video_url'] if 'video_url' in sample else None
    if not url:
        # Fallback if structure varies
        continue

    filename = Path(url).name
    dest = Path(TEST_DIR) / filename
    if not dest.exists():
        try:
            urllib.request.urlretrieve(url, dest)
        except Exception:
            continue

print(f"Downloaded {len(list(Path(TEST_DIR).glob('*.avi')))} test videos.")

# ── Your Original Transformation Logic ──
class DummyCfg:
    img_size = 224
    seed = 42
    num_workers = 0
    debug = True
    pin_memory = False
cfg = DummyCfg()

frame_tf = Tv2.Compose([
    Tv2.ToImage(),
    Tv2.Resize(cfg.img_size, interpolation=InterpolationMode.BILINEAR, antialias=True),
    Tv2.CenterCrop(cfg.img_size),
    Tv2.ToDtype(torch.float32, scale=True),
    Tv2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ── Your Original Dataset & DataLoader Logic (Reads from the test folder) ──
class KineticsSubsetDataset(Dataset):
    def __init__(self, video_dir=TEST_DIR, t=8, img_size=224):
        # Adjusted to grab .avi files which ucf-101 uses
        self.clips    = sorted(Path(video_dir).glob("*.avi"))
        self.t        = t
        self.img_size = img_size
        assert self.clips, f"No clips found in {video_dir}"

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        try:
            container = av.open(str(self.clips[idx]))
            stream    = container.streams.video[0]
            total     = stream.frames or self.t * 4
            start     = _random.randint(0, max(0, total - self.t))
            wanted    = set(range(start, start + self.t))
            frames    = []
            for fi, frame in enumerate(container.decode(video=0)):
                if fi in wanted:
                    img = frame.to_image().convert("RGB").resize((self.img_size, self.img_size))
                    frames.append(frame_tf(img))
                if fi >= start + self.t:
                    break
            container.close()
        except Exception:
            frames = []
        while len(frames) < self.t:
            frames.append(frames[-1].clone() if frames else torch.zeros(3, self.img_size, self.img_size))
        video = torch.stack(frames[:self.t])
        return {"video": video, "augmented_video": video + 0.05 * torch.randn_like(video)}

# Test it out!
dataset = KineticsSubsetDataset(video_dir=TEST_DIR, t=8, img_size=224)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

batch = next(iter(loader))
print(f"\nSuccess! Batch shape: {batch['video'].shape}") # Expected: [2, 8, 3, 224, 224]

🚀 Downloading a small video subset to test your pipeline...


README.md:   0%|          | 0.00/349 [00:00<?, ?B/s]

v_BabyCrawling_g19_c02.avi:   0%|          | 0.00/310k [00:00<?, ?B/s]

v_BasketballDunk_g14_c06.avi:   0%|          | 0.00/546k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2 [00:00<?, ? examples/s]

IndexError: Invalid key: 2 is out of bounds for size 2

## 8. Optimizer

In [ ]:
optimizer = torch.optim.AdamW(
    anyup.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

# Cosine LR decay over the full training run
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.max_steps,
    eta_min=cfg.lr * 0.01,
)

if cfg.resume and Path(cfg.resume).exists():
    ckpt = torch.load(cfg.resume, map_location="cpu", weights_only=False)
    if "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])

print("Optimizer and scheduler ready.")

## 9. Checkpoint Helpers

In [ ]:
ckpt_dir = Path(cfg.checkpoint_dir)
ckpt_dir.mkdir(parents=True, exist_ok=True)

def save_checkpoint(step: int, data_epoch: int, tag: str = ""):
    """
    data_epoch: how many full passes through the dataset have been completed.
    Saved in the checkpoint so that on resume the DataLoader resumes with
    the correct (next) shuffle order and does not replay already-seen data.
    """
    name = f"anyup3d_step{step}{('_' + tag) if tag else ''}.pth"
    path = ckpt_dir / name
    torch.save({
        "anyup": anyup.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "global_step": step,
        "data_epoch":  data_epoch,
        "experiment_name": cfg.experiment_name,
        "cfg": vars(cfg),
    }, path)
    print(f"Saved checkpoint: {path}")
    return path

print(f"Experiment : {cfg.experiment_name}")
print(f"Checkpoints: {ckpt_dir}")

## 10. Training Loop

The loop implements:
1. **T-curriculum** — batch size and frame count change at milestone steps
2. **Primary loss** — `Cosine_MSE` between AnyUp output and teacher GT features
3. **Input-consistency** — same video, different resolution → features should match
4. **Self-consistency** — augmented video → features should be close to clean
5. **Temporal consistency** — adjacent frames → features should be smooth (ramped in)

**Data ordering across sessions:** The DataLoader is seeded with `cfg.seed + data_epoch`.
Each time the iterator runs out of data (one full pass through the dataset), `data_epoch`
increments and the next pass gets a different shuffle seed. This value is saved in every
checkpoint and restored on resume, so after reconnecting the model always continues
into data it has not seen in the same shuffle order.

In [ ]:
# ---- Utility: extract teacher features from a video batch ----
@torch.no_grad()
def extract_teacher_features(video: torch.Tensor) -> torch.Tensor:
    """
    video  : (B, T, C, H, W) float32, ImageNet-normalized
    returns: (B, C_enc, T_eff, H_tok, W_tok)  where T_eff = T // tubelet_size
             e.g. VideoMAE base (tubelet_size=2): T=8 frames -> T_eff=4 temporal tokens
    """
    B, T, C, H, W = video.shape

    # VideoMAE HuggingFace API expects (B, T, C, H, W) — do NOT permute to (B, C, T, H, W).
    pixel_values = video.to(device)
    out = encoder(pixel_values=pixel_values)

    tokens = out.last_hidden_state  # (B, N_tokens, C_enc)
    N = tokens.shape[1]

    # VideoMAE uses tubelet embeddings: temporal tokens = T // tubelet_size
    tubelet_size = encoder.config.tubelet_size
    T_eff = T // tubelet_size
    H_tok = W_tok = int((N / T_eff) ** 0.5)
    assert H_tok * W_tok * T_eff == N, (
        f"Cannot reshape {N} tokens into ({T_eff}, {H_tok}, {W_tok}). "
        f"T={T}, tubelet_size={tubelet_size}, T_eff={T_eff}"
    )

    feats = tokens.reshape(B, T_eff, H_tok, W_tok, -1).permute(0, 4, 1, 2, 3).contiguous()
    return feats  # (B, C_enc, T_eff, H_tok, W_tok)


# ---- Utility: determine current curriculum stage ----
def get_current_stage(step: int) -> TStage:
    stage = cfg.t_schedule[0]
    for s in cfg.t_schedule:
        if step >= s.start_step:
            stage = s
    return stage


print("Training utilities ready.")


In [ ]:
# ---- Main training loop ----

max_steps = cfg.debug_steps if cfg.debug else cfg.max_steps
current_stage = get_current_stage(global_step)

# data_epoch was restored from checkpoint in Cell 5 (or is 0 for a fresh run).
# Pass it to build_loader so the shuffle seed is correct from the start.
loader     = build_loader(t=current_stage.t, batch_size=current_stage.batch_size, epoch=data_epoch)
loader_iter = iter(loader)

anyup.train()

pbar = tqdm(range(global_step, max_steps), initial=global_step, total=max_steps, desc="Training")

for step in pbar:

    # ---- Check if curriculum stage changed ----
    new_stage = get_current_stage(step)
    if new_stage.t != current_stage.t or new_stage.batch_size != current_stage.batch_size:
        print(f"\n[Step {step}] Curriculum: T={current_stage.t}->{new_stage.t}, "
              f"BS={current_stage.batch_size}->{new_stage.batch_size}")
        current_stage = new_stage
        # Stage change resets the loader; bump data_epoch so the new pass is
        # independently shuffled from what was seen before.
        data_epoch += 1
        loader     = build_loader(t=current_stage.t, batch_size=current_stage.batch_size, epoch=data_epoch)
        loader_iter = iter(loader)

    # ---- Get batch (reset iterator when the dataset is exhausted) ----
    try:
        batch = next(loader_iter)
    except StopIteration:
        # One full pass through the data is done. Increment data_epoch so the
        # next pass is seeded differently — this is the key to not replaying
        # the same shuffle order after a session disconnect and resume.
        data_epoch += 1
        loader     = build_loader(t=current_stage.t, batch_size=current_stage.batch_size, epoch=data_epoch)
        loader_iter = iter(loader)
        batch      = next(loader_iter)

    video     = batch["video"].to(device)           # (B, T, C, H, W)
    aug_video = batch.get("augmented_video")
    if aug_video is not None:
        aug_video = aug_video.to(device)

    B, T, C, H, W = video.shape

    # ---- Forward pass ----
    with autocast(device_type=device.type, dtype=torch.bfloat16):

        gt_feats  = extract_teacher_features(video)          # (B, C_enc, T, H_tok, W_tok)
        video_5d  = video.permute(0, 2, 1, 3, 4)            # (B, C, T, H, W)
        out_size  = (gt_feats.shape[-2], gt_feats.shape[-1]) # (H_tok, W_tok)
        pred_feats = anyup(video_5d, gt_feats, output_size=out_size)

        # 1. Primary reconstruction loss
        loss_rec = cosine_mse_3d(pred_feats, gt_feats) * cfg.lambda_cos_mse

        # 2. Self-consistency under augmentation
        loss_aug = torch.tensor(0.0, device=device)
        if aug_video is not None and cfg.lambda_self_consistency > 0:
            aug_5d = aug_video.permute(0, 2, 1, 3, 4)
            with torch.no_grad():
                gt_feats_aug = extract_teacher_features(aug_video)
            pred_aug = anyup(aug_5d, gt_feats_aug, output_size=out_size)
            loss_aug = cosine_mse_3d(pred_aug, pred_feats.detach()) * cfg.lambda_self_consistency

        # 3. Temporal consistency
        loss_temp = torch.tensor(0.0, device=device)
        t_lam = get_temporal_lambda(step)
        if T > 1 and t_lam > 0:
            diff = pred_feats[:, :, 1:] - pred_feats[:, :, :-1]
            loss_temp = diff.pow(2).mean() * t_lam

        total_loss = loss_rec + loss_aug + loss_temp

    # ---- Backward ----
    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(anyup.parameters(), cfg.grad_clip_max_norm)
    optimizer.step()
    scheduler.step()

    # ---- Logging ----
    if step % cfg.log_every_n_steps == 0:
        lr = optimizer.param_groups[0]["lr"]
        writer.add_scalar("loss/total",    total_loss.item(), step)
        writer.add_scalar("loss/rec",      loss_rec.item(),   step)
        writer.add_scalar("loss/aug",      loss_aug.item(),   step)
        writer.add_scalar("loss/temporal", loss_temp.item(),  step)
        writer.add_scalar("lr",            lr,                step)
        writer.add_scalar("curriculum/T",  current_stage.t,   step)
        writer.add_scalar("curriculum/bs", current_stage.batch_size, step)
        writer.add_scalar("data/epoch",    data_epoch,        step)

        pbar.set_postfix({
            "loss":       f"{total_loss.item():.4f}",
            "rec":        f"{loss_rec.item():.4f}",
            "T":          current_stage.t,
            "lr":         f"{lr:.2e}",
            "data_epoch": data_epoch,
        })

    # ---- Checkpoint ----
    if (step + 1) % cfg.save_every_n_steps == 0:
        save_checkpoint(step + 1, data_epoch)

    if cfg.debug and step >= cfg.debug_steps - 1:
        print(f"Debug mode: stopping after {cfg.debug_steps} steps.")
        break


# Final checkpoint
save_checkpoint(max_steps, data_epoch, tag="final")
writer.flush()
writer.close()
print("Training complete.")

## 11. Quick Validation / Smoke Test

Run this at any point to check the model produces sensible outputs without running the full training loop.

In [ ]:
anyup.eval()

with torch.no_grad():
    dummy_video = torch.randn(1, 4, 3, cfg.img_size, cfg.img_size, device=device)
    dummy_feats = torch.randn(1, ENCODER_DIM, 4, TOKEN_H, TOKEN_W, device=device)
    dummy_video_5d = dummy_video.permute(0, 2, 1, 3, 4)

    out = anyup(dummy_video_5d, dummy_feats, output_size=(TOKEN_H, TOKEN_W))
    print(f"Input  features : {dummy_feats.shape}")
    print(f"Output features : {out.shape}")
    assert out.shape == dummy_feats.shape, "Shape mismatch!"
    print("Validation passed.")

anyup.train()